Item based colaborative filtering

In [20]:
import pandas as pd
import numpy as np


df = pd.read_csv('reviews (1).csv').sample(n=5000, random_state=1)
apps_df = pd.read_csv('applications.csv')

df = pd.merge(df, apps_df[['appid', 'name']], on='appid', how='left')

cf_data = df[['author_steamid', 'name', 'voted_up']].copy()

cf_data = cf_data.dropna(subset=['name'])

cf_data['rating'] = cf_data['voted_up'].astype(int)
cf_data = cf_data.groupby(['author_steamid', 'name'])['rating'].max().reset_index()

global_mean = cf_data['rating'].mean()

item_means = cf_data.groupby('name')['rating'].mean()
item_bias = item_means - global_mean

user_means = cf_data.groupby('author_steamid')['rating'].mean()
user_bias = user_means - global_mean

def get_baseline(user, item):
    if user in user_bias:
        bias_u = user_bias[user]
    else:
        bias_u = 0
    if item in item_bias:
        bias_i = item_bias[item]
    else:
        bias_i = 0

    return global_mean + bias_u + bias_i


def calculate_cosine_similarity(item_i, item_j, data):
    users_i = data[data['name'] == item_i].set_index('author_steamid')['rating']
    users_j = data[data['name'] == item_j].set_index('author_steamid')['rating']

    common_users = users_i.index.intersection(users_j.index)

    if len(common_users) == 0:
        return 0.0
    dot_product = np.sum(users_i[common_users] * users_j[common_users])
    magnitude_i = np.sqrt(np.sum(users_i**2))
    magnitude_j = np.sqrt(np.sum(users_j**2))

    if magnitude_i == 0 or magnitude_j == 0:
        return 0.0

    return dot_product / (magnitude_i * magnitude_j)

def predict_rating(target_user, target_item, data, k=3):
    base_ui = get_baseline(target_user, target_item)
    user_history = data[data['author_steamid'] == target_user]

    if user_history.empty:
        return max(0.0, min(1.0, base_ui))
    similarities = {}
    for played_item in user_history['name']:
        sim = calculate_cosine_similarity(target_item, played_item, data)
        similarities[played_item] = sim

    top_k = sorted({k:v for k,v in similarities.items() if v > 0}.items(), key=lambda x: x[1], reverse=True)[:k]

    numerator = 0
    denominator = 0

    for played_item, sim in top_k:
        actual_rating = user_history[user_history['name'] == played_item]['rating'].values[0]
        base_uj = get_baseline(target_user, played_item)
        deviation = actual_rating - base_uj

        numerator += sim * deviation
        denominator += sim
    if denominator == 0:
        raw_prediction = base_ui
    else:
        raw_prediction = base_ui + (numerator / denominator)
    final_prediction = max(0.0, min(1.0, raw_prediction))

    return final_prediction

target_game = cf_data['name'].value_counts().idxmax()
target_user = cf_data['author_steamid'].value_counts().idxmax()
print("\nPredicting for User ", target_user, " on ",target_game)
prediction = predict_rating(target_user, target_game, cf_data, k=3)
print("\nPredicted Rating (0 to 1 scale):", prediction)

/tmp/ipykernel_5118/1032232305.py:6: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  apps_df = pd.read_csv('applications.csv')


Calculating Baselines...

Predicting for User  76561198355752399  on  Nature Calls

Predicted Rating (0 to 1 scale): 0.9866


In [2]:
import pandas as pd
import numpy as np
import math


df = pd.read_csv('reviews (1).csv', dtype={'author_steamid': str, 'appid': str}).sample(n=5000, random_state=1)
df['rating'] = df['voted_up'].astype(int)


user_to_game = df.groupby('author_steamid')['appid'].apply(list).to_dict()
game_to_user = df.groupby('appid')['author_steamid'].apply(list).to_dict()


user_game_to_rating = {}
for index, row in df.iterrows():
    user_game_to_rating[(row['author_steamid'], row['appid'])] = row['rating']


global_means = {}
for user, games in user_to_game.items():
    ratings = []
    for game in games:
        ratings.append(user_game_to_rating[(user, game)])
    global_means[user] = np.mean(ratings)


target_user = '76561198237676960'
target_games = set(user_to_game[target_user])
mu_target = global_means[target_user]


similarity_scores = {}
for other_user in user_to_game:
    if other_user == target_user:
        continue

    other_games = set(user_to_game[other_user])
    common_games = target_games.intersection(other_games)


    if len(common_games) >= 3:
        mu_other = global_means[other_user]
        numerator = 0
        den_target = 0
        den_other = 0

        for game in common_games:
            r_t = user_game_to_rating[(target_user, game)]
            r_o = user_game_to_rating[(other_user, game)]

            numerator += (r_t - mu_target) * (r_o - mu_other)
            den_target += (r_t - mu_target)**2
            den_other += (r_o - mu_other)**2


        if den_target > 0 and den_other > 0:
            sim = numerator / (math.sqrt(den_target) * math.sqrt(den_other))
        else:
            sim = 0
        similarity_scores[other_user] = sim


predicted_ratings = {}
for game in game_to_user:
    if game in target_games:
        continue

    raters = game_to_user[game]
    num = 0
    den = 0

    for u in raters:
        if u in similarity_scores:
            sim = similarity_scores[u]
            if sim != 0:
                num += sim * user_game_to_rating[(u, game)]
                den += abs(sim)

    if den > 0:
        predicted_ratings[game] = num / den


recommendations = sorted(predicted_ratings.items(), key=lambda x: x[1], reverse=True)[:5]

print(f"Recommendations for {target_user}:")
for game_id, score in recommendations:
    print(f"Game AppID: {game_id} | Predicted Score: {score:.2f}")

Recommendations for 76561198237676960:
